This notebook documents the exploratory process of acquiring and filtering the raw GEO datasets (GSE114134, GSE114135, GSE34639, GSE189148) — including dead ends and debugging steps, preserved for scientific transparency. It makes live calls to GEO's servers and will re-download data if rerun. The final data products this notebook produces are: GSE189148_sample_metadata.csv, GSE189148_resting_sample_metadata.csv, and the GSE189148 IDAT directory, all outputs the other notebooks build on.

Loading/cleaning datasets

In [1]:
import GEOparse
GEOparse.set_verbosity("WARNING")

In [ ]:
import GEOparse # To read the datasets

accessions = ["GSE59999", "GSE114134", "GSE114135"]

#So we can see the metadata info and figure out what allergic and healthy datapoints are named
for acc in accessions:
    gse = GEOparse.get_GEO(geo=acc, destdir="../data")
    print(acc, "-> samples:", len(gse.gsms), "| platform:", list(gse.gpls.keys()))
    first_sample = list(gse.gsms.values())[0]
    print(first_sample.metadata.get("characteristics_ch1"))
 

/home/ethan-xiao/food-allergy-biomarkers/fa-biomarkers-env/lib/python3.14/site-packages/GEOparse/GEOparse.py:401: DtypeWarning: Columns (0: CHR, 1: Chromosome_36, 2: Coordinate_36, 3: SPOT_ID) have mixed types. Specify dtype option on import or set low_memory=False.
  return read_csv(StringIO(data), index_col=None, sep="\t")


In [ ]:
accessions = ["GSE59999", "GSE114134", "GSE114135"]


for acc in accessions:
    gse = GEOparse.get_GEO(geo=acc, destdir="../data")
    print(f"=== {acc} ===")
    labels_seen = set() # Checking for duplicate labels
    
    for gsm_name, gsm in gse.gsms.items():
        platform = gsm.metadata.get("platform_id", ["unknown"])[0]
        characteristics = tuple(gsm.metadata.get("characteristics_ch1", []))
        labels_seen.add((platform, characteristics))
    
    for entry in list(labels_seen)[:15]:
        print(entry)
    print(f"total unique label combinations: {len(labels_seen)}")
    print("---"), 

In [ ]:
accessions = ["GSE59999", "GSE114134", "GSE114135"]
for acc in accessions:
    gse = GEOparse.get_GEO(geo=acc, destdir="../data")
    print(f"=== {acc} ===")
    for platform_id, gpl in gse.gpls.items():
        print(platform_id)
        print("Title:", gpl.metadata.get("title", ["unknown"])) #Checking potential differences in methylation data
        print("Technology:", gpl.metadata.get("technology", ["unknown"]))
        print("---")

In [ ]:
gse_114135 = GEOparse.get_GEO(geo="GSE114135", destdir="../data")

filtered_gsms = {}  # We're going to filter by technology

for gsm_name, gsm in gse_114135.gsms.items():
    platform = gsm.metadata.get("platform_id", ["unknown"])[0]
    
    if platform == "GPL23976":
        filtered_gsms[gsm_name] = gsm

print("Total samples before filtering:", len(gse_114135.gsms.items()))
print("Total samples after filtering:", len(filtered_gsms))

#Uh oh turns out it may be the same as GSE114134 (probably should have guessed that)

Checking whether GSE114134 and GSE114135 are distinct or the same thing

In [ ]:
gse_114134 = GEOparse.get_GEO(geo="GSE114134", destdir="../data")

ids_114134 = set()
for gsm_name, gsm in gse_114134.gsms.items():
    characteristics = gsm.metadata.get("characteristics_ch1", [])
    print(characteristics)
    break 

In [ ]:
gse_114134 = GEOparse.get_GEO(geo="GSE114134", destdir="../data", how="quick")

one_sample = list(gse_114134.gsms.values())[0]

print("Title:", one_sample.metadata.get("title", ["unknown"]))
print("Full metadata:", one_sample.metadata) #Just so I can see everything going on

In [ ]:
datasets_to_check = ["GSE114135", "GSE34639", "GSE189148"] #Checking for similarities, ages, sample numbers

for acc in datasets_to_check:
    gse = GEOparse.get_GEO(geo=acc, destdir="../data", how="quick")
    print(f"=== {acc} ===")
    
    platform_counts = {}
    age_values = set()
    
    for gsm_name, gsm in gse.gsms.items():
        platform = gsm.metadata.get("platform_id", ["unknown"])[0]
        platform_counts[platform] = platform_counts.get(platform, 0) + 1
        
        characteristics = gsm.metadata.get("characteristics_ch1", [])
        for item in characteristics:
            if "age" in item.lower():
                age_values.add(item)
    
    print("Sample counts by platform:", platform_counts)
    print("All distinct age-related values found:", age_values)
    
    one_sample = list(gse.gsms.values())[0]
    print("Example source_name_ch1 (cell type):", one_sample.metadata.get("source_name_ch1", ["unknown"]))
    print("---")

Checking for RAW IDAT 

In [ ]:
gse_189148 = GEOparse.get_GEO(geo="GSE189148", destdir="../data", how="quick") #We're going to use this one instead

one_sample = list(gse_189148.gsms.values())[0]
print(one_sample.metadata)

In [ ]:
datasets_to_check = ["GSE114134", "GSE34639"] #Checking to see if these guys have raw sample IDATs

for acc in datasets_to_check:
    gse = GEOparse.get_GEO(geo=acc, destdir="../data", how='quick')
    print(f"=== {acc} ===")
    
    # Check series level
    print("Series-level supplementary file:", gse.metadata.get('supplementary_file', ['unknown']))
    
    # Check all samples supplementary files
    file_values = set()
    for gsm_name, gsm in gse.gsms.items():
        supp = tuple(gsm.metadata.get('supplementary_file', ['unknown']))
        file_values.add(supp)
    
    print("All distinct supplementary_file values across samples:", file_values)
    print("---")

In [ ]:
for acc in datasets_to_check:
    gse = GEOparse.get_GEO(geo=acc, destdir="../data", how='quick')
    print(f"=== {acc} ===")
    
    one_sample = list(gse.gsms.values())[0]
    print(one_sample.metadata)
    print("---")

Filtering for Rested Values

In [ ]:
datasets_to_check = ["GSE114134", "GSE34639", "GSE189148"] #Seeing what the names might be for activated v resting

for acc in datasets_to_check:
    gse = GEOparse.get_GEO(geo=acc, destdir="../data", how="quick")
    print(f"=== {acc} ===")
    
    condition_values = set()
    for gsm_name, gsm in gse.gsms.items():
        characteristics = gsm.metadata.get("characteristics_ch1", [])
        for item in characteristics:
            if any(word in item.lower() for word in ["stimul", "treatment", "activate"]):
                condition_values.add(item)
    
    print("All condition-related values found:", condition_values)
    print("---")

In [ ]:
def filter_by_condition(gse, field_name, keep_value): #Only keeping the resting (quiescient) ones
    filtered_gsms = {}
    for gsm_name, gsm in gse.gsms.items():
        characteristics = gsm.metadata.get("characteristics_ch1", [])
        target = f"{field_name}: {keep_value}"
        if target in characteristics:
            filtered_gsms[gsm_name] = gsm
    return filtered_gsms

gse_114134 = GEOparse.get_GEO(geo="GSE114134", destdir="../data", how="quick")
gse_34639 = GEOparse.get_GEO(geo="GSE34639", destdir="../data", how="quick")
gse_189148 = GEOparse.get_GEO(geo="GSE189148", destdir="../data", how="quick")

resting_114134 = filter_by_condition(gse_114134, "stimulation", "0")
resting_34639 = filter_by_condition(gse_34639, "treatment", "CON")
resting_189148 = filter_by_condition(gse_189148, "stimulation_status", "U")

print("GSE114134 resting samples:", len(resting_114134))
print("GSE34639 resting samples:", len(resting_34639))
print("GSE189148 resting samples:", len(resting_189148))

Filtering for Timepoint

In [ ]:
for acc, gse in [("GSE114134", gse_114134), ("GSE34639", gse_34639), ("GSE189148", gse_189148)]: #Looking at first few samples' characteristics to find timepoint
    print(f"=== {acc} ===")
    for gsm_name, gsm in list(gse.gsms.items())[:6]:
        title = gsm.metadata.get("title", ["unknown"])
        characteristics = gsm.metadata.get("characteristics_ch1", [])
        print(title, "|", characteristics)
    print("---")

In [ ]:
print("=== GSE34639 control-group samples ===")
for gsm_name, gsm in gse_34639.gsms.items():
    characteristics = gsm.metadata.get("characteristics_ch1", [])
    if any("disease state: CONTROL" in item or "disease state: CON" in item or "disease state: NON" in item.upper() for item in characteristics):
        title = gsm.metadata.get("title", ["unknown"])
        print(title, "|", characteristics) #Trying to isolate the control group

In [ ]:
disease_states = set()
for gsm_name, gsm in gse_34639.gsms.items():
    characteristics = gsm.metadata.get("characteristics_ch1", [])
    for item in characteristics:
        if "disease" in item.lower():
            disease_states.add(item)

print(disease_states) #Maybe this way?

In [ ]:
print("=== GSE34639: Food Allergy group ===") #Creative name, I know
for gsm_name, gsm in gse_34639.gsms.items():
    characteristics = gsm.metadata.get("characteristics_ch1", [])
    if "disease state: FOOD ALLERGY" in characteristics:
        title = gsm.metadata.get("title", ["unknown"])
        print(title, "|", characteristics)

print("=== GSE34639 Control group ===") #
for gsm_name, gsm in gse_34639.gsms.items():
    characteristics = gsm.metadata.get("characteristics_ch1", [])
    if "disease state: NORM" in characteristics:
        title = gsm.metadata.get("title", ["unknown"])
        print(title, "|", characteristics)

In [ ]:
earliest_applicable_114134 = filter_by_condition(gse_114134, "age", "1")
earliest_applicable_34639 = filter_by_condition(gse_34639, "age", "0")

# keep only samples that are resting and earliest timepoint
final_114134 = {name: gsm for name, gsm in resting_114134.items() if name in earliest_applicable_114134}
final_34639 = {name: gsm for name, gsm in resting_34639.items() if name in earliest_applicable_34639}

print("GSE114134 final (resting + earliest):", len(final_114134))
print("GSE34639 final (resting + earliest):", len(final_34639))
print("GSE189148 final (resting, single timepoint already):", len(resting_189148))

Checking for IDATs/beta values

In [ ]:
gse_34639_full = GEOparse.get_GEO(geo="GSE34639", destdir="../data", how="full")

one_sample = list(gse_34639_full.gsms.values())[0] #Checking format of sample data
print(one_sample.table.head())
print(one_sample.table.shape)

In [ ]:
print(one_sample.metadata.get("data_processing", ["unknown"])) #Verifying that they've been normalized 
print("Min:", one_sample.table['VALUE'].min())
print("Max:", one_sample.table['VALUE'].max())

In [ ]:
import gc #It crashed on me so
del gse_34639_full
gc.collect()

In [ ]:
gse_114134_full = GEOparse.get_GEO(geo="GSE114134", destdir="../data", how="full")
gse_189148_full = GEOparse.get_GEO(geo="GSE189148", destdir="../data", how="full") #Checking the other two now

for name, gse in [("GSE114134", gse_114134_full), ("GSE189148", gse_189148_full)]:
    one_sample = list(gse.gsms.values())[0]
    print(f"=== {name} ===")
    print(one_sample.table.head())
    print("Min:", one_sample.table.iloc[:, 1].min())
    print("Max:", one_sample.table.iloc[:, 1].max())
    print("---")

In [ ]:
one_sample = list(gse_114134_full.gsms.values())[0] #Looking at the data
print(one_sample.table.head())
print("Shape:", one_sample.table.shape)
print("Columns:", list(one_sample.table.columns))

In [ ]:
print("=== GSE114134 range ===")
print("Min:", gse_114134_full.gsms[list(gse_114134_full.gsms.keys())[0]].table['VALUE'].min())
print("Max:", gse_114134_full.gsms[list(gse_114134_full.gsms.keys())[0]].table['VALUE'].max())

In [ ]:
import gc
del gse_114134_full #Next one
gc.collect()

gse_189148_full = GEOparse.get_GEO(geo="GSE189148", destdir="../data", how="full")
one_sample = list(gse_189148_full.gsms.values())[0]
print("=== GSE189148 ===")
print(one_sample.table.head())
print("Min:", one_sample.table['VALUE'].min())
print("Max:", one_sample.table['VALUE'].max())

In [ ]:
# Check if data is stored at series level 
print("Series-level tables available:", list(gse_189148_full.gpls.keys()))

# Try to assemble a combined matrix
try:
    matrix = gse_189148_full.pivot_samples('VALUE')
    print("pivot worked, shape:", matrix.shape)
    print(matrix.head())
except Exception as e:
    print("pivot_samples failed:", e)

In [ ]:
# Check for a series supplementary matrix file
print("Series supplementary files:")
print(gse_189148_full.metadata.get('supplementary_file', ['none']))

#Check for sample-level supplementary files
one_sample = list(gse_189148_full.gsms.values())[0]
print("\nSample supplementary files:")
print(one_sample.metadata.get('supplementary_file', ['none']))

In [ ]:
gse_189148_full.download_supplementary_files(
    directory="../data/GSE189148_suppl",
    download_sra=False
)

In [ ]:
import urllib.request #Let's just get it from here, GEOparse didn't work
import os

os.makedirs("../data/GSE189148_suppl", exist_ok=True)

url = "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE189nnn/GSE189148/suppl/GSE189148_matrix_processed.csv.gz"
dest = "../data/GSE189148_suppl/GSE189148_matrix_processed.csv.gz"

urllib.request.urlretrieve(url, dest)
print("Downloaded:", os.path.getsize(dest), "bytes")

In [ ]:
import pandas as pd

#Load the first few rows to see the structure without loading the whole thing
preview = pd.read_csv("../data/GSE189148_suppl/GSE189148_matrix_processed.csv.gz", nrows=5)
print("Shape of preview:", preview.shape)
print("Columns:", list(preview.columns)[:10])  #first 10 column names
print(preview.iloc[:, :5])  #first 5 columns of the preview

In [ ]:
#Check if the SOFT metadata gives a column order or supplementary sample label that we can use to map the numbers to the gsms
one_sample = list(gse_189148_full.gsms.values())[0]
print("Title:", one_sample.metadata.get("title", ["?"]))
print("Description:", one_sample.metadata.get("description", ["?"]))
print("Supplementary:", one_sample.metadata.get("supplementary_file", ["?"]))
print("---")
#Also check the second sample for a pattern 
second = list(gse_189148_full.gsms.values())[1]
print("2nd title:", second.metadata.get("title", ["?"]))
print("2nd description:", second.metadata.get("description", ["?"]))

In [ ]:
import urllib.request
import os

url = "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE189nnn/GSE189148/matrix/GSE189148_series_matrix.txt.gz"
dest = "../data/GSE189148_suppl/GSE189148_series_matrix.txt.gz"

try:
    urllib.request.urlretrieve(url, dest)
    print("Downloaded series matrix:", os.path.getsize(dest), "bytes")
except Exception as e:
    print("Failed:", e)

In [ ]:
#List the GSMs in the order GEOparse loaded them with identifying info
for i, (gsm_name, gsm) in enumerate(gse_189148_full.gsms.items(), start=1):
    title = gsm.metadata.get("title", ["?"])[0]
    print(f"Position {i}: {gsm_name} | {title}")
    if i >= 10:  # just the first 10 for now
        break

In [ ]:
#Load the CSV's probe IDs & Sample 1's value column
csv_data = pd.read_csv(
    "../data/GSE189148_suppl/GSE189148_matrix_processed.csv.gz",
    usecols=["Unnamed: 0", "Sample 1_Methylatedsignal"]
)
csv_data.columns = ["probe", "sample1_value"]
csv_data = csv_data.set_index("probe")
print("Sample 1 values at a few probes:")
print(csv_data.head())
print("Total probes:", len(csv_data))

In [ ]:
import pandas as pd

#Read header row to get column names
header = pd.read_csv(
    "../data/GSE189148_suppl/GSE189148_matrix_processed.csv.gz",
    nrows=0
)
value_cols = [c for c in header.columns if "Methylatedsignal" in c]
print("Number of value columns:", len(value_cols))
print("First 15 in file order:")
for c in value_cols[:15]:
    print(" ", c)

In [ ]:
one_sample = list(gse_189148_full.gsms.values())[0]  #GSM5695305 = Single_FA_1_stim full dump
print("GSM5695305 — ALL metadata keys and values:")
for key, value in one_sample.metadata.items():
    print(f"{key}: {value}")

In [ ]:
import re

records = []
for gsm_name, gsm in gse_189148_full.gsms.items():
    title = gsm.metadata.get("title", ["?"])[0]
    supp = gsm.metadata.get("supplementary_file", [""])[0]
    #pull barcode out of the IDAT filename
    match = re.search(r'(\d{12}_R\d{2}C\d{2})', supp)
    barcode = match.group(1) if match else "NONE"
    records.append((gsm_name, title, barcode))

#sort by barcode to see if it forms a usable sequence
for gsm_name, title, barcode in sorted(records, key=lambda x: x[2])[:20]:
    print(barcode, "|", gsm_name, "|", title)

In [ ]:
import re

records = []
for gsm_name, gsm in gse_189148_full.gsms.items():
    chars = gsm.metadata.get("characteristics_ch1", [])
    sex = next((c.split(":")[1].strip() for c in chars if c.lower().startswith("sex")), "?")
    supp = gsm.metadata.get("supplementary_file", [""])[0]
    match = re.search(r'(\d{12}_R\d{2}C\d{2})', supp)
    barcode = match.group(1) if match else "NONE"
    records.append({"gsm": gsm_name, "sex": sex, "barcode": barcode})

#This is annoying, but let's use sex to check correlation: 
#candidate ordering A: GSM/submission order
gsm_order_sex = [r["sex"] for r in records]
#candidate ordering B: barcode order
barcode_order_sex = [r["sex"] for r in sorted(records, key=lambda x: x["barcode"])]

print("GSM-order sex sequence:    ", "".join("M" if s=="M" else "F" for s in gsm_order_sex))
print("Barcode-order sex sequence:", "".join("M" if s=="M" else "F" for s in barcode_order_sex))

In [ ]:
import pandas as pd
import numpy as np

#Load full CSV value columns (probes + samples)
#We need the _Methylatedsignal columns + probe IDs
usecols = ["Unnamed: 0"] + [c for c in header.columns if "Methylatedsignal" in c]
df = pd.read_csv(
    "../data/GSE189148_suppl/GSE189148_matrix_processed.csv.gz",
    usecols=usecols
).set_index("Unnamed: 0")
df.columns = [c.replace("_Methylatedsignal", "") for c in df.columns]  # -> "Sample 1", etc.
print("Loaded matrix:", df.shape)  # (probes, 88 samples)

In [ ]:
#Reduce to columns sorted numerically, so "Sample N" lines up with position N-1
#Build ordered list of the sample column names by their numeric part
def sample_num(colname):
    return int(colname.replace("Sample ", ""))

ordered_cols = sorted(df.columns, key=sample_num)  # Sample 1, Sample 2, ... Sample 88
df = df[ordered_cols]  # reorder columns numerically

#gsm_order_sex  and  barcode_order_sex  were built earlier
#Testing if metadata-sex labeling produces clean sex separation in the data?

def separation_score(sex_labels):
    labels = np.array([1 if s == "M" else 0 for s in sex_labels])
    #mean methylation per probe in each group
    male_mean = df.loc[:, labels == 1].mean(axis=1)
    female_mean = df.loc[:, labels == 0].mean(axis=1)
    diff = (male_mean - female_mean).abs()
    #how many probes show a large (>0.3 beta) male/female difference?
    strong_probes = (diff > 0.3).sum()
    return strong_probes

print("Strong sex-diff probes under GSM order:    ", separation_score(gsm_order_sex))
print("Strong sex-diff probes under barcode order:", separation_score(barcode_order_sex))

In [ ]:
import numpy as np

#For each probe, compute variance across samples; high-variance probes on sex chromosomes
#should show clear bimodal (two-cluster) structure if sex signal exists (which it should according to the internet)
probe_var = df.var(axis=1)
top_var_probes = probe_var.sort_values(ascending=False).head(20)
print("Top 20 most variable probes:")
print(top_var_probes)

#Look at the single most variable probe's values across all 88 samples
most_var = top_var_probes.index[0]
vals = df.loc[most_var].sort_values()
print(f"\nSorted values for most variable probe ({most_var}):")
print(vals.values)

In [ ]:
#Count sexes in metadata
from collections import Counter
meta_sex_counts = Counter(r["sex"] for r in records)
print("Metadata sex counts:", meta_sex_counts)

In [ ]:
import numpy as np

#Bimodal probe cleanly splits high/low at ~0.5. Using 0.9/0.1 threshold to call sex per column.

probe_vals = df.loc["cg00445548"]
#38 samples are "high", 50 are "low"; metadata has 51 M / 37 F
#"high" (38) likely = F (37), "low" (50) likely = M (51)  (double check tho)
bio_sex = probe_vals.apply(lambda v: "F" if v > 0.5 else "M")
print("Biological sex call counts:", bio_sex.value_counts().to_dict())

#bio_sex is indexed by "Sample N". We need to line up metadata records to Sample N via each ordering
def agreement(records_in_order):
    #records_in_order: list of metadata sexes assumed to correspond to Sample 1..88 in numeric order
    matches = sum(1 for i, sex in enumerate(records_in_order, start=1)
                  if bio_sex.get(f"Sample {i}") == sex)
    return matches, len(records_in_order)

print("GSM order agreement:    ", agreement(gsm_order_sex))
print("Barcode order agreement:", agreement(barcode_order_sex))

In [ ]:
gse_189148_full = GEOparse.get_GEO(geo="GSE189148", destdir="../data", how="full")

for gsm_id in ['GSM5695309', 'GSM5695325']:
    print(gsm_id)
    print(gse_189148_full.gsms[gsm_id].metadata['characteristics_ch1'])
    print()

In [ ]:
#List first 10 GSMs
for i, (gsm_name, gsm) in enumerate(gse_189148_full.gsms.items(), start=1):
    title = gsm.metadata.get("title", ["?"])[0]
    print(f"Position {i}: {gsm_name} | {title}")
    if i >= 10: 
        break

Building actual metadata table

In [ ]:
records = []
for gsm_id, gsm_obj in gse_189148_full.gsms.items():
    chars = gsm_obj.metadata['characteristics_ch1']
    record = {'gsm': gsm_id}
    for entry in chars:
        key, value = entry.split(': ', 1)
        if key in record:
            key = key + '_detailed' #If sample has two same, only changes second one
        record[key] = value
    records.append(record)

metadata_df = pd.DataFrame(records)
metadata_df.to_csv('../data/GSE189148_sample_metadata.csv', index=False)
metadata_df.head()

In [ ]:
print(len(metadata_df))
print(metadata_df['gsm'].nunique())

In [ ]:
idat_dir = '../data/GSE189148_idats'
files = os.listdir(idat_dir)

basenames = {}
for f in files:
    match = re.match(r'(GSM\d+)_(\d+_R\d+C\d+)_(Grn|Red)\.idat\.gz', f) #Bc there's two channels in methylation array data
    if match:
        gsm_id = match.group(1)
        barcode_position = match.group(2)
        basenames[gsm_id] = os.path.join(idat_dir, gsm_id + '_' + barcode_position)

metadata_df['Basename'] = metadata_df['gsm'].map(basenames)
metadata_df['Basename'].isna().sum()

In [ ]:
metadata_df.to_csv('../data/GSE189148_sample_metadata.csv', index=False)

In [ ]:
resting_df = metadata_df[metadata_df['stimulation_status'] == 'U'] #Applying the filter
len(resting_df)

In [ ]:
resting_df.to_csv('../data/GSE189148_resting_sample_metadata.csv', index=False)

Looking at a few things

In [5]:
import pandas as pd

resting_df = pd.read_csv('/home/ethan-xiao/food-allergy-biomarkers/data/GSE189148_resting_sample_metadata.csv')
len(resting_df)

resting_df['Basename'].apply(lambda x: x.split('/')[-1].split('_')[1])

0     204088100082
1     204088100082
2     204281260061
3     204273120074
4     204273120073
5     204281260061
6     204088100071
7     204281260063
8     204088100053
9     204273120074
10    204281260060
11    204281260061
12    204281260061
13    204273120066
14    204281260063
15    204273120005
16    204088100071
17    204273120074
18    204273120068
19    204088100053
20    204281260061
21    204281260060
22    204273120074
23    204273120074
24    204273120068
25    204273120066
26    204273120074
27    204273120005
28    204088100071
29    204281260043
30    204088100071
31    204281260043
32    204273120073
33    204281260043
34    204281260063
35    204273120068
36    204273120068
37    204088100082
38    204281260063
39    204273120074
40    204281260063
41    204273120066
42    204273120073
Name: Basename, dtype: str

In [6]:
resting_df['Basename'].apply(lambda x: x.split('/')[-1].split('_')[1]).value_counts()

Basename
204273120074    7
204281260061    5
204281260063    5
204088100071    4
204273120068    4
204088100082    3
204273120073    3
204273120066    3
204281260043    3
204088100053    2
204281260060    2
204273120005    2
Name: count, dtype: int64